In [0]:
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
landing_path = f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Check whether the landing path (and its parent directories) exists and contains files
try:
    path_has_files = len(dbutils.fs.ls(landing_path)) > 0
except Exception:
    path_has_files = False

# Exit outside the try block so the exit signal is not swallowed by the except clause
if not path_has_files:
    print("No new files are received")
    dbutils.notebook.exit("No new files are received")

if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")